# 03. KNN 회귀

가까운 데이터의 값을 이용해 연속형 값을 예측하는 **K-최근접 이웃 회귀(KNN Regression)** 를 학습합니다.

### 이번 노트에서 배우는 것
- `KNeighborsRegressor`
- 회귀에서 `score()`가 의미하는 R²
- MAE / MSE로 오차 확인
- `n_neighbors` 튜닝
- KNN이 훈련 범위 밖에서 가지는 한계


In [ ]:
from common.libraries import *


## 1. 데이터 준비

NumPy 배열 생성, `train_test_split`, `reshape`는 이미 배운 준비 과정이므로 한 셀에 묶습니다.
KNN은 입력을 `(행, 특성)` 형태의 2차원 데이터로 받기 때문에 길이 데이터를 `reshape(-1, 1)`로 바꿉니다.


In [ ]:
perch_length = np.array([8.4, 13.7, 15.0, 16.2, 17.4, 18.0, 18.7, 19.0, 19.6, 20.0, 21.0, 21.0, 21.0, 21.3, 22.0, 22.0, 22.0, 22.0, 22.0, 22.5, 22.5, 22.7, 23.0, 23.5, 24.0, 24.0, 24.6, 25.0, 25.6, 26.5, 27.3, 27.5, 27.5, 27.5, 28.0, 28.7, 30.0, 32.8, 34.5, 35.0, 36.5, 36.0, 37.0, 37.0, 39.0, 39.0, 39.0, 40.0, 40.0, 40.0, 40.0, 42.0, 43.0, 43.0, 43.5, 44.0])
perch_weight = np.array([5.9, 32.0, 40.0, 51.5, 70.0, 100.0, 78.0, 80.0, 85.0, 85.0, 110.0, 115.0, 125.0, 130.0, 120.0, 120.0, 130.0, 135.0, 110.0, 130.0, 150.0, 145.0, 150.0, 170.0, 225.0, 145.0, 188.0, 180.0, 197.0, 218.0, 300.0, 260.0, 265.0, 250.0, 250.0, 300.0, 320.0, 514.0, 556.0, 840.0, 685.0, 700.0, 700.0, 690.0, 900.0, 650.0, 820.0, 850.0, 900.0, 1015.0, 820.0, 1100.0, 1000.0, 1100.0, 1000.0, 1000.0])

train_input, test_input, train_target, test_target = train_test_split(
    perch_length,
    perch_weight,
    test_size=0.25,
    random_state=42,
)

train_input = train_input.reshape(-1, 1)
test_input = test_input.reshape(-1, 1)

print("Train:", train_input.shape, train_target.shape)
print("Test:", test_input.shape, test_target.shape)


## 2. 데이터 관계 확인

산점도로 길이가 증가할수록 무게도 증가하는 경향이 있는지 확인합니다.


In [ ]:
plt.scatter(perch_length, perch_weight)
plt.xlabel("length")
plt.ylabel("weight")
plt.title("농어 길이와 무게")
plt.show()


## 3. KNN 회귀 모델 학습

KNN 회귀는 예측하려는 값과 가까운 `K`개의 훈련 데이터를 찾고,
그 이웃들의 정답값을 이용해 예측합니다.

회귀 모델의 `score()`는 **정확도(accuracy)가 아니라 R²(결정계수)** 를 반환합니다.


In [ ]:
knr = KNeighborsRegressor(n_neighbors=5)
knr.fit(train_input, train_target)

train_r2 = knr.score(train_input, train_target)
test_r2 = knr.score(test_input, test_target)
test_pred = knr.predict(test_input)

print("훈련 R²:", round(train_r2, 4))
print("테스트 R²:", round(test_r2, 4))
print("MAE:", round(mean_absolute_error(test_target, test_pred), 3))
print("MSE:", round(mean_squared_error(test_target, test_pred), 3))


## 4. `n_neighbors` 튜닝

`n_neighbors`는 사람이 정하는 **하이퍼파라미터**입니다.

- 너무 작으면 주변 몇 개 데이터에 지나치게 맞춰져 과적합될 수 있음
- 너무 크면 너무 많은 이웃을 평균내 과소적합될 수 있음

여기서는 여러 값을 비교하고 테스트 R²가 가장 높은 값을 확인합니다.


In [ ]:
results = []

for n in range(1, 21):
    model = KNeighborsRegressor(n_neighbors=n)
    model.fit(train_input, train_target)

    results.append({
        "n_neighbors": n,
        "train_r2": model.score(train_input, train_target),
        "test_r2": model.score(test_input, test_target),
    })

knn_results = pd.DataFrame(results)
display(knn_results.sort_values("test_r2", ascending=False).head())

best_n = int(knn_results.loc[knn_results["test_r2"].idxmax(), "n_neighbors"])
print("선택된 n_neighbors:", best_n)


## 5. 선택한 모델과 이웃 확인

`kneighbors()`를 사용하면 특정 값을 예측할 때 어떤 훈련 데이터를 이웃으로 사용했는지 확인할 수 있습니다.


In [ ]:
knr = KNeighborsRegressor(n_neighbors=best_n)
knr.fit(train_input, train_target)

sample_length = 50
sample_pred = knr.predict([[sample_length]])[0]
distances, indexes = knr.kneighbors([[sample_length]])

print(f"길이 {sample_length} 예측 무게: {sample_pred:.2f}")

plt.scatter(train_input[:, 0], train_target, label="train")
plt.scatter(train_input[indexes, 0], train_target[indexes], marker="D", label="neighbors")
plt.scatter(sample_length, sample_pred, marker="^", s=100, label="prediction")
plt.legend()
plt.show()


## 6. KNN의 한계

KNN은 **훈련 데이터 안에서 가까운 이웃을 찾는 모델**입니다.
따라서 훈련 데이터의 최대 길이를 훨씬 벗어난 값을 입력해도 같은 이웃들이 반복해서 선택될 수 있고,
예측값이 더 이상 증가하지 않는 문제가 생깁니다.


In [ ]:
for length in [50, 100, 1000]:
    pred = knr.predict([[length]])[0]
    _, idx = knr.kneighbors([[length]])
    print(f"길이 {length:>4} → 예측 {pred:>8.2f} / 사용 이웃 index: {idx[0]}")


### 정리
- KNN 회귀는 가까운 이웃의 정답값을 이용해 예측합니다.
- 회귀에서 `score()`는 R²입니다.
- `n_neighbors`에 따라 과적합/과소적합 정도가 달라집니다.
- 훈련 데이터 범위를 크게 벗어난 값을 예측하는 데 약점이 있어 다음 노트에서 선형·다항 회귀를 학습합니다.
